In [ ]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [3]:
# LangChain
# Main framework used to build LLM and RAG applications.

# langchain-core
# Contains the core building blocks of LangChain such as Documents, Prompts, Messages, and Runnables.

# langchain-community
# Provides integrations with third-party tools, document loaders, vector databases, and other services.

# pypdf
# Used to read and extract text from PDF files.

# pymupdf
# PDF processing library used for extracting text, images, tables, and page information.

# sentence-transformers
# Used to convert text into embeddings (vectors) for semantic search.

# chromadb
# Vector database used to store embeddings and perform similarity/semantic search.

### To create a sample document

In [4]:
from langchain_core.documents import Document

In [5]:
sample_document = Document(
    page_content = "Hello World ! My name is Aryan",
    metadata = {"source":"apnacollege"}
)

In [8]:
print(sample_document)
print(type(sample_document))

page_content='Hello World ! My name is Aryan' metadata={'source': 'apnacollege'}
<class 'langchain_core.documents.base.Document'>


### Loading data from txt file

In [2]:
from langchain_community.document_loaders.text import TextLoader

In [10]:
txt_laoder = TextLoader("Data/Python.txt",encoding ="utf-8")

In [13]:
document = txt_laoder.load()
print(document)

[Document(metadata={'source': 'Data/Python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

### Loading data from pdf

In [15]:
from langchain_community.document_loaders.pdf import PyPDFLoader , PyMuPDFLoader # for simple data , for complex data

In [ ]:
pdf_loader = PyPDFLoader("Data/research2.pdf")
document = pdf_loader.load()
print(document)

# Ingestion pipeline

### Converting data into documents

In [3]:
import os 
from langchain_community.document_loaders.pdf import PyPDFLoader

In [4]:
def load_all_docs():
    folder_path = "data/pdfs"
    num_of_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # Complete path
            complete_path = os.path.join(folder_path,filename)
            pdf_loader = PyPDFLoader(complete_path)
            doc = pdf_loader.load()

            all_docs.extend(doc)
            num_of_docs += 1

    print(f"Number of docs = {num_of_docs}")
    print(f"Number of docs = {len(all_docs)}")
    return all_docs

In [5]:
all_pdf_documents = load_all_docs()

Number of docs = 2
Number of docs = 32


### Converting Document into multiple Chunks

In [6]:
!pip install langchain_text_splitters

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
def split_data(documents , chunk_size = 500 , chunk_overlap = 50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [9]:
chunks = split_data(all_pdf_documents)

### Embedding

In [10]:
from sentence_transformers import SentenceTransformer

#### Embedding Manager

In [11]:
class EmbeddingManager:
    def __init__(self, model_name = "all-MiniLM-L6-v2"):

        self.model_name = model_name
        print(f"Loading model... {self.model_name}")
        self.model = SentenceTransformer(self.model_name)
        print(f"Embedding Dimensions : {self.model.get_sentence_embedding_dimension()}")


    def generate_embedding(self,text):
        embeddings = self.model.encode(text , show_progress_bar = True)
        print(f"Embedding shape : ", embeddings.shape)
        return embeddings

In [12]:
Embedding_Manager = EmbeddingManager()

Loading model... all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1563.18it/s]


Embedding Dimensions : 384


C:\Users\hacke\AppData\Local\Temp\ipykernel_23164\2210317196.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding Dimensions : {self.model.get_sentence_embedding_dimension()}")


### Vector Store

In [13]:
import chromadb
import uuid

In [14]:
class VectorStoreManager:
    def __init__(self , persist_directory = "data/vector_store" , collection_name = "pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    # Vector store extra information (meta-data)
    def _initialize_store(self):
        # Creating directory 
        os.makedirs(self.persist_directory, exist_ok = True)

        # Create a client
        self.client = chromadb.PersistentClient(path = self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata = {"description" : "vector store collection for pdf embeddings in RAG"}
        )

        print(f"Initialized the vector store with collection {self.collection_name}")
        print(f"docs in collection : {self.collection.count()}")

    # To actually add data in collection
    def add_documents(self , documents ,embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents does not match num of embeddings")


        # Store => ids , embedding , document , metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i , (doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)
            
            embeddings_list.append(embedding)

            self.collection.add(
                ids = ids,
                metadatas = all_metadata,
                documents = documents_content,
                embeddings = embeddings_list
            )

        print(f"Total documents added in vector store = {len(documents_content)}")
        print(f"Docs in collection = {self.collection.count()}")

In [15]:
vector_store = VectorStoreManager()

Initialized the vector store with collection pdf_documents
docs in collection : 642


### Creating embeddings

In [16]:
text = [doc.page_content for doc in chunks]

In [17]:
embeddings = Embedding_Manager.generate_embedding(text)

Batches: 100%|██████████| 11/11 [00:28<00:00,  2.62s/it]

Embedding shape :  (321, 384)


### Storing data in vector store 

In [18]:
vector_store.add_documents(chunks , embeddings)

Total documents added in vector store = 321
Docs in collection = 963


# Retrieve Pipeline

In [19]:
class RAGRetriever:
    def __init__(self , Embedding_Manager , vector_store):
        self.Embedding_Manager = Embedding_Manager
        self.vector_store = vector_store


    def retrieve(self , query , top_k = 5 , score_threshold = 0.0):
        # Query => Embedding
        query_embeddings = self.Embedding_Manager.generate_embedding([query])[0]

        # Semantic Search
        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k
        )

        # Cosine similarity
        retrieved_docs = []
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id , metadata , document , distance) in enumerate(zip(ids , metadatas , documents ,distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id" : doc_id,
                        "document" : document,
                        "metadata" : metadata,
                        "distance" : distance,
                        "similarity_score" : similarity_score,
                        "rank" : i+1
                    })

            print(f"Retrieved : {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [20]:
rag_retriever = RAGRetriever(Embedding_Manager,vector_store)

In [27]:
result = rag_retriever.retrieve("What is RAG?")

Batches: 100%|██████████| 1/1 [00:00<00:00, 53.47it/s]

Embedding shape :  (1, 384)
Retrieved : 5 documents


In [38]:

for res in result:
    doc = res["document"]
    print(doc)

and speculate on upcoming trends and innovations.
Our contributions are as follows:
• In this survey, we present a thorough and systematic
review of the state-of-the-art RAG methods, delineating
its evolution through paradigms including naive RAG,
arXiv:2312.10997v5  [cs.CL]  27 Mar 2024
and speculate on upcoming trends and innovations.
Our contributions are as follows:
• In this survey, we present a thorough and systematic
review of the state-of-the-art RAG methods, delineating
its evolution through paradigms including naive RAG,
arXiv:2312.10997v5  [cs.CL]  27 Mar 2024
and speculate on upcoming trends and innovations.
Our contributions are as follows:
• In this survey, we present a thorough and systematic
review of the state-of-the-art RAG methods, delineating
its evolution through paradigms including naive RAG,
arXiv:2312.10997v5  [cs.CL]  27 Mar 2024
effective RAG framework.
• We have summarized the current assessment methods of
RAG, covering 26 tasks, nearly 50 datasets, outlining

# Integration with LLMs

### OpenAI - GPT

In [76]:
from dotenv import load_dotenv
import os

load_dotenv(".env")

True

In [24]:
!pip install langchain-openai

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.7 MB 799.2 kB/s eta 0:00:02
   ------------ --------------------------- 0.5/1.7 MB 799.2 kB/s eta 0:00:02
   ------------------ --------------------- 0.8/1.7 MB 644.9 kB/s eta 0:00:02
   ------------------ --------------------- 0.8/1.7 MB 644.9 kB/s eta 0:00:02
   ------------------ --------------------- 0.8/1.7 MB 644.9 kB/s eta 0:00:02
   ------------------------ --------------- 1.0/1.7 MB 565.4 kB/s eta 0:00:02
   ------------------------------ --------- 1.3/1.7 MB 633.2 kB/s eta 0:00:01
   ------------------------------------ --- 1.6/1.7 MB 699.0 kB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 745.1 kB/s  0:00:02
   ----------------------

In [25]:
from langchain_openai import ChatOpenAI

In [41]:
!pip install python-dotenv

In [77]:
llm = ChatOpenAI(
    openai_api_key = os.getenv("API_KEY_OPENAI"),
    model = "gpt-5.4",
    temperature = 0.1,
    max_tokens = 1024
)

### Generate our retrieval-augmented output

In [39]:
def generate_output(query , rag_retriever , llm , top_k = 3):
    results = rag_retriever.retrieve(query,top_k)
    context = ""
    if results:
        for result in results:
            doc = result["document"]
            context += doc +  "\n\n"

    # Context + query 
    prompt = f""" use given context to generate the answer for the query 
    Context : {context}
    Query : {query}"""

    response = llm.invoke(prompt) # Expects a string as prompt
    return response

In [ ]:
answer = generate_output("What is RAG?", rag_retriever,llm)

### GROQ

In [46]:
!pip install langchain-groq


   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 2/2 [langchain-groq]



In [47]:
from langchain_groq import ChatGroq

In [78]:
llm = ChatGroq(
    groq_api_key = os.getenv("API_KEY_GROQ"),
    model = "qwen/qwen3.6-27b",
    temperature = 0.1,
    max_tokens = 1024
)

In [63]:
def generate_output(query , rag_retriever , llm , top_k = 3):
    results = rag_retriever.retrieve(query,top_k)
    context = ""
    if results:
        for result in results:
            doc = result["document"]
            context += doc +  "\n\n"

    # Context + query 
    prompt = f""" use given context to generate the answer for the query 
    Context : {context}
    Query : {query}"""

    # response = llm.invoke([prompt.format(context = context, query = query)]) # Expects a string as prompt
    response = llm.invoke(prompt)
    return response

In [64]:
answer = generate_output("What is RAG?", rag_retriever,llm)

Batches: 100%|██████████| 1/1 [00:00<00:00, 50.21it/s]


Embedding shape :  (1, 384)
Retrieved : 3 documents


In [65]:
print(answer)

content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - **Context:** The provided context is repetitive and fragmented. It mentions:\n     - "and speculate on upcoming trends and innovations."\n     - "Our contributions are as follows:"\n     - "• In this survey, we present a thorough and systematic review of the state-of-the-art RAG methods, delineating its evolution through paradigms including naive RAG,"\n     - "arXiv:2312.10997v5 [cs.CL] 27 Mar 2024"\n     - This repeats three times.\n   - **Query:** "What is RAG?"\n\n2.  **Identify Key Information in Context:**\n   - The context mentions "RAG methods" and "naive RAG" as part of a survey on state-of-the-art methods.\n   - It references an arXiv paper: `arXiv:2312.10997v5 [cs.CL] 27 Mar 2024`.\n   - However, the context *does not explicitly define* what RAG stands for or what it is. It only mentions it as a subject of a survey/review.\n\n3.  **Determine Constraints & Requirements:**\n   - The prompt say